# Data Preparation & Random Forest — a worked example

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/SCAI-4EUWorkshopAIinMedicineWorkshop/blob/main/Hands-On-Session-1/data_preparation_example.ipynb)

## Objective

Real-world medical data is rarely clean — it has **missing values**, **outliers**, and
**categorical variables** that all need preprocessing before any model can learn from it.

Here we work with a noised version of the **Breast Cancer Wisconsin** dataset (diagnose a
tumor as *malignant* or *benign*) and walk through the full workflow:

> **Explore → Visualize → Prepare → Train → Evaluate**

1. Discover the data's quality problems.
2. Learn the **data-preparation pipeline**: imputation → outlier handling → encoding → scaling.
3. Train a **Random Forest** and compare it against a logistic-regression baseline.

**Key idea:** on messy data, careful preparation usually matters more than the choice of model.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay)

## 2. Load & Explore the Data

This dataset is built to look like **real clinical data**:

- some measurements are **missing** (equipment errors, incomplete records),
- some contain **outliers** (data-entry mistakes, extreme cases),
- there are **categorical** columns (patient demographics).

Your first job as a data scientist is to **discover and understand these issues** before any
modeling.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/racousin/SCAI-4EUWorkshopAIinMedicineWorkshop/main/Hands-On-Session-1/breast_cancer_noisy.csv")

# Target: 0 = malignant, 1 = benign
target_names = np.array(["malignant", "benign"])

# Split columns by type (the target is excluded from the feature lists)
numeric_cols     = df.select_dtypes(include=[np.number]).columns.drop("target").tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()

print(f"Shape: {df.shape}")
print(f"{len(numeric_cols)} numeric features")
print(f"{len(categorical_cols)} categorical features: {categorical_cols}")
df.head()

Use `.info()` (types + non-null counts) and `.describe()` (value ranges) to spot the three
classic problems: **missing values**, **outliers** (suspiciously large max values), and
**categorical** (non-numeric) columns.

In [ ]:
df.info()
df.describe()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

## 3. Visualize the Data Issues

Before fixing problems, we need to **see** them.

### 3.1 — Where are the missing values?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(df[numeric_cols].isnull(), cbar=False, yticklabels=False, ax=axes[0], cmap="viridis")
axes[0].set_title("Missing-value map (numeric features)")
axes[0].tick_params(axis="x", rotation=90)

df[numeric_cols].isnull().sum().plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Missing values per feature")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### 3.2 — Outliers

Boxplots show the median, the quartiles, and **outliers** (points beyond 1.5×IQR). `mean
radius` and `mean area` have a clear tail of extreme values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df["mean radius"], ax=axes[0], color="lightblue")
axes[0].set_title("Mean radius — raw (with outliers)")
sns.boxplot(y=df["mean area"], ax=axes[1], color="lightgreen")
axes[1].set_title("Mean area — raw (with outliers)")
plt.tight_layout()
plt.show()

### 3.3 — Distribution of the categorical features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, categorical_cols, ["steelblue", "salmon", "mediumseagreen"]):
    df[col].value_counts().plot(kind="bar", ax=ax, color=color)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 4. Handle Missing Values

Most models cannot train on `NaN`, so we fill the gaps with `SimpleImputer`:

| Strategy | When to use |
|---|---|
| Drop rows | Very few missing values, large dataset |
| **Median** imputation | Numeric features (robust to outliers) |
| **Most-frequent** imputation | Categorical features |

We use **median** for numeric columns (robust to the outliers we just saw) and
**most-frequent** for categorical columns.

In [ ]:
num_imputer = SimpleImputer(strategy="median")
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

cat_imputer = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

print("Total missing values after imputation:", int(df.isnull().sum().sum()))

## 5. Handle Outliers

Outliers can distort training. We detect them with the **IQR method** and **clip** them to
the bounds:

- **IQR** = Q3 − Q1
- **lower** = Q1 − 1.5·IQR, **upper** = Q3 + 1.5·IQR
- values outside `[lower, upper]` are set to the nearest bound.

**Why clip instead of drop?** In medical data every patient matters — clipping keeps the
sample while taming the extreme value.

In [ ]:
print("Before clipping:")
print(f"  mean radius  max = {df['mean radius'].max():.2f}")
print(f"  mean area    max = {df['mean area'].max():.2f}")

for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

print("\nAfter clipping:")
print(f"  mean radius  max = {df['mean radius'].max():.2f}")
print(f"  mean area    max = {df['mean area'].max():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df["mean radius"], ax=axes[0], color="lightblue")
axes[0].set_title("Mean radius — after clipping")
sns.boxplot(y=df["mean area"], ax=axes[1], color="lightgreen")
axes[1].set_title("Mean area — after clipping")
plt.tight_layout()
plt.show()

## 6. Encode Categorical Features

Models work with **numbers**, not text. **One-hot encoding** creates one binary column per
category:

| tumor_location | → | central | lower_inner | upper_inner |
|---|---|---|---|---|
| upper_inner | | 0 | 0 | 1 |
| central     | | 1 | 0 | 0 |

**Why not just number the categories (0, 1, 2)?** That would imply an **order** between them,
which usually isn't meaningful. `drop_first=True` removes one redundant column per feature
(the *dummy-variable trap*).

In [ ]:
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
new_cols = [c for c in df.columns if c not in numeric_cols and c != "target"]
print(f"Shape after encoding: {df.shape}")
print(f"New one-hot columns: {new_cols}")
df.head()

## 7. Split & Scale

Separate features `X` from the target `y`, hold out 20% for testing, then standardize each
feature to mean 0 / std 1:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

**Crucial:** fit the scaler on the **training set only**, then apply it to both — otherwise
information from the test set leaks into training.

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train
X_test_scaled  = scaler.transform(X_test)        # apply to test

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]} samples")

## 8. Baseline: Logistic Regression

Always start with a **simple, well-understood model**. Its score is the reference any fancier
model must beat.

In [ ]:
lr = LogisticRegression(max_iter=5000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print(f"Logistic Regression accuracy: {accuracy_score(y_test, y_pred_lr):.4f}\n")
print(classification_report(y_test, y_pred_lr, target_names=target_names))

## 9. Random Forest

A **Random Forest** is an **ensemble** of many **decision trees**, each trained on a random
subset of the rows and features. The final prediction is the **majority vote** of all trees;
because the trees make *different* errors, the errors tend to cancel out.

| Property | Logistic Regression | Random Forest |
|---|---|---|
| Decision boundary | Linear | Non-linear |
| Sensitive to scale / outliers | Yes | No |
| Interpretability | Coefficients | Feature importances |

> **Medical analogy:** like asking several doctors — the consensus is usually more reliable
> than any single opinion.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print(f"Logistic Regression accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Random Forest accuracy:       {accuracy_score(y_test, y_pred_rf):.4f}\n")
print(classification_report(y_test, y_pred_rf, target_names=target_names))

### 9.1 — Compare the confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels=target_names,
                                        cmap="Blues", ax=axes[0])
axes[0].set_title("Logistic Regression")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, display_labels=target_names,
                                        cmap="Greens", ax=axes[1])
axes[1].set_title("Random Forest")
plt.tight_layout()
plt.show()

### 9.2 — Feature importances

A Random Forest reports how much each feature contributed — valuable for understanding *what
drives the diagnosis*.

In [ ]:
importances = rf.feature_importances_
order = np.argsort(importances)[-15:]

plt.figure(figsize=(9, 7))
plt.barh(range(len(order)), importances[order], color="steelblue")
plt.yticks(range(len(order)), X.columns[order])
plt.xlabel("Feature importance")
plt.title("Top 15 features (Random Forest)")
plt.tight_layout()
plt.show()

## 10. Hyperparameter Tuning

Hyperparameters are settings we choose **before** training. `GridSearchCV` tries every
combination and keeps the best one, scored by cross-validation.

| Hyperparameter | Controls | Too low | Too high |
|---|---|---|---|
| `n_estimators` | Number of trees | Unstable | Slow, diminishing returns |
| `max_depth` | Tree depth | Underfit | Overfit (memorizes noise) |
| `min_samples_split` | Samples needed to split | Overfit | Underfit |

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
}
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")

y_pred_best = grid.best_estimator_.predict(X_test_scaled)
print(f"\nDefault RF test accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Tuned   RF test accuracy: {accuracy_score(y_test, y_pred_best):.4f}")

## 11. Cross-Validation Comparison

A single train/test split can be lucky or unlucky. **5-fold cross-validation** rotates the
test fold through all the data for a more reliable estimate. We wrap scaling and the model in
a `Pipeline` so the scaler is re-fit on each fold's training portion only — **no leakage**.

In [ ]:
lr_pipe = Pipeline([("scale", StandardScaler()),
                    ("model", LogisticRegression(max_iter=5000, random_state=42))])
rf_pipe = Pipeline([("scale", StandardScaler()),
                    ("model", grid.best_estimator_)])

lr_scores = cross_val_score(lr_pipe, X, y, cv=5, scoring="accuracy")
rf_scores = cross_val_score(rf_pipe, X, y, cv=5, scoring="accuracy")

print(f"{'Model':<24}{'Accuracy':>10}{'Std':>10}")
print("-" * 44)
print(f"{'Logistic Regression':<24}{lr_scores.mean():>10.4f}{lr_scores.std():>10.4f}")
print(f"{'Random Forest (tuned)':<24}{rf_scores.mean():>10.4f}{rf_scores.std():>10.4f}")

## 12. Summary

| Step | Technique | Why |
|---|---|---|
| Missing values | `SimpleImputer` (median / most-frequent) | Fill `NaN` without dropping samples |
| Outliers | IQR clipping | Tame extreme values, keep the sample |
| Categorical encoding | One-hot (`get_dummies`) | Turn text categories into numbers |
| Feature scaling | `StandardScaler` | Put features on a common scale |
| Baseline | Logistic Regression | A reference score to beat |
| Stronger model | Random Forest | Ensemble of trees — captures non-linear patterns |
| Tuning | `GridSearchCV` | Find good hyperparameters via cross-validation |

```
Raw data → impute → handle outliers → encode → scale → split → train → evaluate
```

**Key insight:** data preparation is often more important than model choice. A well-prepared
dataset with a simple model frequently beats a complex model on messy data.